In [1]:
print("Hello")

Hello


In [2]:
from embedder import Embedder

embed = Embedder()


2026-06-29 20:05:17.107927207 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


Q1.Embedding a query

In [3]:
q='How does approximate nearest neighbor search work?'
v = embed.encode(q)
v[0]

np.float64(-0.02058203437252893)

Q2. Cosine Similarity

In [4]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [5]:
for d in documents:
    if d["filename"] == "02-vector-search/lessons/07-sqlitesearch-vector.md":
        doc = d
        break
doc

{'content': '# Vector Search with sqlitesearch\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=csxKescwJYM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous section we used minsearch for vector search.\n\nIt works, but it has three problems:\n\n- It rebuilds the index on every startup\n- It keeps everything in memory\n- It searches by brute force\n\n\nWith text search we never felt these. Indexing was fast because we\ndidn\'t embed anything. With vector search, indexing runs a neural\nnetwork over every document, so it takes a minute on our dataset.\nKeeping everything in memory is fine here, but a larger dataset would\nneed too much space.\n\nThe third problem is brute-force search. For every query we compare the\nquery vector against every single document. With 1,000 documents this is\nfine, probably even faster than anything smarter. But as the dataset\ngrows past 10,000 or so, it gets slow, and we\'ll want an approximate\nmethod instead.\n\nWhat we\'ve done 

In [6]:
import numpy as np
# Query from Q1
query = "How does approximate nearest neighbor search work?"

# Create embeddings
query_vec = embed.encode(query)
doc_vec = embed.encode(doc["content"])

# Cosine similarity (vectors are already normalized)
similarity = np.dot(query_vec, doc_vec)

print(similarity)

0.36107027225589694


Q3.Chunking and search by hand

In [7]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)
X = embed.encode_batch([chunk["content"] for chunk in chunks])
scores = X.dot(v)
best_idx = scores.argmax()
print(chunks[best_idx]["filename"])

02-vector-search/lessons/07-sqlitesearch-vector.md


Q4.Vector search with minsearch

In [8]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["filename"])

vindex.fit(X, chunks)

query = "What metric do we use to evaluate a search engine?"
query_vector = embed.encode(query)

results = vindex.search(query_vector, num_results=5)

print(results[0]["filename"])


04-evaluation/lessons/05-search-metrics.md


5.Text search vs vector search

In [9]:
q1 = "How do I store vectors in PostgreSQL?"

In [10]:
#Vector search
query_vector = embed.encode(q1)
results = vindex.search(query_vector, num_results=5)
print(results)

[{'start': 0, 'content': '# Vector Search with PGVector\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=0P54MFyz-mc&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nMany real databases can do vector search. Elasticsearch has it, and\nthere are dedicated stores like Qdrant and Chroma. We\'ll go with\nPostgres. Most of us already run it at work, and the data engineering\ncourse uses it too. The concept is the same as with sqlitesearch, only\nthe database under the hood changes.\n\n[pgvector](https://github.com/pgvector/pgvector) is the PostgreSQL\nextension that makes this work. Install it and Postgres can do vector\nsimilarity search. On top of that you get the usual production features,\nlike concurrent access, transactions, and large datasets.\n\nWe\'ll run Postgres with pgvector in Docker.\n\n## Starting Postgres with pgvector\n\nPull the image and start a container:\n\n```bash\ndocker run -it \\\n    --name pgvector \\\n    -e POSTGRES_USER=user \\\n    -e POSTGRES_PASSWORD

In [11]:

#text search
from minsearch import Index

# Create and fit the index
index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
index.fit(chunks)

In [12]:
text_results = index.search(q1, num_results=5)
text_results

[{'start': 4000,
  'content': 'get 0.01.\n\nThe first score for `q1` vs `d` (0.32) is higher, so that query is more\nsimilar to the document about registration. The second score for `q2`\nvs `d` sits near 0, because installing Docker has nothing to do with\nregistration. A score near 0 means the two vectors are about as\ndifferent as they can be.\n\nThat\'s the whole idea behind vector search: similar texts get similar\nvectors, and a dot product tells us how similar.\n\n## Cosine similarity\n\nThe `all-MiniLM-L6-v2` model outputs normalized vectors - vectors with\nunit length. When both vectors are normalized, the dot product equals\ncosine similarity. That\'s why the model documentation says it "uses\ncosine similarity."\n\nCosine similarity measures the angle between two vectors, ignoring\ntheir length:\n\n- 1.0 = same direction (similar)\n- 0.0 = perpendicular (unrelated)\n- -1.0 = opposite direction (opposite meaning)\n\nFormally, if `theta` is the angle between two vectors, cosin

In [13]:
# Filenames from text search
text_files = {r["filename"] for r in text_results}

# Filenames from vector search
vector_files = {r["filename"] for r in results}

# Files that appear only in vector search
vector_only = vector_files - text_files

print(vector_only)

{'02-vector-search/lessons/08-pgvector.md'}


6.Hybrid Search

In [14]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [15]:
q2="How do I give the model access to tools?"

In [17]:
# Index
text_results = index.search(q2, num_results=5)

# Vector
v2 = embed.encode(q2)
vector_results = vindex.search(v2, num_results=5)

In [18]:
hybrid_results = rrf([vector_results, text_results])
hybrid_results[0]["filename"]

'01-agentic-rag/lessons/13-function-calling.md'